In [1]:
# ================================================================
# 02_PREPROCESSING.PY
# Anticipatory Solar Forecasting Research
# Clean data + engineer all features
# ================================================================

import pandas as pd
import numpy as np
import os

os.makedirs("data/processed", exist_ok=True)

# ----------------------------------------------------------------
# 1. CITY-SPECIFIC CLEAR SKY MAXIMUMS
# (same values as original paper)
# ----------------------------------------------------------------
CLEAR_SKY_MAX = {
    "Riyadh":  1000,
    "Jeddah":  980,
    "Abha":    950,
    "Dammam":  990,
    "Qassim":  995,
}

# ----------------------------------------------------------------
# 2. PREPROCESSING FUNCTION
# ----------------------------------------------------------------
def preprocess_city(city_name):
    print(f"\n{'='*55}")
    print(f"  Processing: {city_name}")
    print(f"{'='*55}")

    # --- Load raw data ---
    path = f"data/raw/{city_name.lower()}_raw.csv"
    df   = pd.read_csv(path)

    # --- Parse timestamp ---
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df["timestamp"] = df["timestamp"].dt.tz_convert("Asia/Riyadh")

    # ── TEMPORAL FEATURES ───────────────────────────────────────
    df["hour"]        = df["timestamp"].dt.hour
    df["month"]       = df["timestamp"].dt.month
    df["day_of_year"] = df["timestamp"].dt.dayofyear
    df["year"]        = df["timestamp"].dt.year
    df["season"]      = df["month"].map({
        1:"Winter", 2:"Winter", 3:"Spring",
        4:"Spring", 5:"Spring", 6:"Summer",
        7:"Summer", 8:"Summer", 9:"Autumn",
        10:"Autumn",11:"Autumn",12:"Winter"
    })

    # ── FILTER DAYLIGHT HOURS (05:00–19:00) ─────────────────────
    # Same as original paper
    df = df[(df["hour"] >= 5) & (df["hour"] <= 19)].copy()
    print(f"  ✓ After daylight filter: {len(df):,} rows")

    # ── PAPER'S ORIGINAL FEATURES ───────────────────────────────
    # Reproduce exactly what the paper used
    df["hour_x_cloud"]    = df["hour"] * df["cloud_cover"]
    df["temp_x_humidity"] = df["temperature"] * df["humidity"]

    # ── CLEAR SKY RATIO ─────────────────────────────────────────
    max_ghi = CLEAR_SKY_MAX[city_name]
    df["clear_sky_ratio"] = df["GHI"] / max_ghi
    df["clear_sky_ratio"] = df["clear_sky_ratio"].clip(0, 1)

    # ── NEW FEATURES (OUR CONTRIBUTION) ─────────────────────────

    # -- Dust proxy features --
    # High wind + low humidity = dust storm conditions
    df["dust_proxy"] = df["wind_speed"] * (1 - df["humidity"] / 100)

    # Wind direction: convert to sin/cos
    # (so that 359° and 1° are treated as close, not far apart)
    df["wind_sin"] = np.sin(np.radians(df["wind_dir"]))
    df["wind_cos"] = np.cos(np.radians(df["wind_dir"]))

    # Northerly wind in Riyadh = dust from Arabian Desert
    df["wind_speed_x_cloud"] = df["wind_speed"] * df["cloud_cover"]

    # -- Atmospheric instability --
    # Rapid pressure drop = incoming weather event
    df["pressure_diff"] = df["pressure"].diff().fillna(0)

    # -- Rolling trend features (3-hour window) --
    # These are KEY for our meta-model — they capture
    # atmospheric conditions CHANGING over time
    df["cloud_trend_3h"]    = df["cloud_cover"].rolling(3).mean().fillna(method="bfill")
    df["wind_trend_3h"]     = df["wind_speed"].rolling(3).mean().fillna(method="bfill")
    df["humidity_trend_3h"] = df["humidity"].rolling(3).mean().fillna(method="bfill")
    df["GHI_trend_3h"]      = df["GHI"].rolling(3).mean().fillna(method="bfill")

    # -- Rate of change features --
    # How fast is cloud cover INCREASING? 
    # Fast increase = approaching storm
    df["cloud_change_1h"] = df["cloud_cover"].diff().fillna(0)
    df["wind_change_1h"]  = df["wind_speed"].diff().fillna(0)
    df["GHI_change_1h"]   = df["GHI"].diff().fillna(0)

    # ── DATA SPLIT LABELS ────────────────────────────────────────
    # Train: 2019-2022 | Val: 2023 | Test: 2024
    def assign_split(year):
        if year <= 2022: return "train"
        elif year == 2023: return "val"
        else: return "test"

    df["split"] = df["year"].apply(assign_split)

    # ── DROP NULLS ───────────────────────────────────────────────
    before = len(df)
    df = df.dropna()
    after  = len(df)
    print(f"  ✓ Dropped {before - after} rows with nulls")
    print(f"  ✓ Final rows: {after:,}")

    # ── SPLIT SUMMARY ────────────────────────────────────────────
    split_counts = df["split"].value_counts()
    print(f"  ✓ Train rows: {split_counts.get('train', 0):,}")
    print(f"  ✓ Val rows:   {split_counts.get('val', 0):,}")
    print(f"  ✓ Test rows:  {split_counts.get('test', 0):,}")

    # ── SEASON SUMMARY ───────────────────────────────────────────
    season_counts = df["season"].value_counts()
    print(f"  ✓ Seasons: {dict(season_counts)}")

    # ── SAVE ────────────────────────────────────────────────────
    out_path = f"data/processed/{city_name.lower()}_processed.csv"
    df.to_csv(out_path, index=False)
    print(f"  ✓ Saved: {out_path}")

    return df


# ----------------------------------------------------------------
# 3. PROCESS ALL CITIES
# ----------------------------------------------------------------
all_processed = {}

for city in ["Riyadh", "Jeddah", "Abha", "Dammam", "Qassim"]:
    all_processed[city] = preprocess_city(city)


# ----------------------------------------------------------------
# 4. FINAL SUMMARY
# ----------------------------------------------------------------
print(f"\n{'='*55}")
print("  PREPROCESSING COMPLETE")
print(f"{'='*55}")
print(f"\n  Features created:")
print(f"  PAPER FEATURES:")
print(f"    hour, month, day_of_year, cloud_cover,")
print(f"    temperature, humidity, hour_x_cloud,")
print(f"    temp_x_humidity, clear_sky_ratio")
print(f"\n  OUR NEW FEATURES:")
print(f"    dust_proxy, wind_sin, wind_cos,")
print(f"    wind_speed_x_cloud, pressure_diff,")
print(f"    cloud_trend_3h, wind_trend_3h,")
print(f"    humidity_trend_3h, GHI_trend_3h,")
print(f"    cloud_change_1h, wind_change_1h,")
print(f"    GHI_change_1h")
print(f"\n  Data split:")
print(f"    Train → 2019-2022 (4 years)")
print(f"    Val   → 2023      (1 year)")
print(f"    Test  → 2024      (1 year)")


  Processing: Riyadh
  ✓ After daylight filter: 32,880 rows
  ✓ Dropped 0 rows with nulls
  ✓ Final rows: 32,880
  ✓ Train rows: 21,915
  ✓ Val rows:   5,475
  ✓ Test rows:  5,490
  ✓ Seasons: {'Spring': 8280, 'Summer': 8280, 'Autumn': 8190, 'Winter': 8130}


C:\Users\redaj\AppData\Local\Temp\ipykernel_19792\3959326269.py:89: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["cloud_trend_3h"]    = df["cloud_cover"].rolling(3).mean().fillna(method="bfill")
C:\Users\redaj\AppData\Local\Temp\ipykernel_19792\3959326269.py:90: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["wind_trend_3h"]     = df["wind_speed"].rolling(3).mean().fillna(method="bfill")
C:\Users\redaj\AppData\Local\Temp\ipykernel_19792\3959326269.py:91: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["humidity_trend_3h"] = df["humidity"].rolling(3).mean().fillna(method="bfill")
C:\Users\redaj\AppData\Local\Temp\ipykernel_19792\3959326269.py:92: FutureWarning: Series.fillna with 'method' is deprecated and will raise 

  ✓ Saved: data/processed/riyadh_processed.csv

  Processing: Jeddah
  ✓ After daylight filter: 32,880 rows
  ✓ Dropped 0 rows with nulls
  ✓ Final rows: 32,880
  ✓ Train rows: 21,915
  ✓ Val rows:   5,475
  ✓ Test rows:  5,490
  ✓ Seasons: {'Spring': 8280, 'Summer': 8280, 'Autumn': 8190, 'Winter': 8130}


C:\Users\redaj\AppData\Local\Temp\ipykernel_19792\3959326269.py:89: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["cloud_trend_3h"]    = df["cloud_cover"].rolling(3).mean().fillna(method="bfill")
C:\Users\redaj\AppData\Local\Temp\ipykernel_19792\3959326269.py:90: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["wind_trend_3h"]     = df["wind_speed"].rolling(3).mean().fillna(method="bfill")
C:\Users\redaj\AppData\Local\Temp\ipykernel_19792\3959326269.py:91: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["humidity_trend_3h"] = df["humidity"].rolling(3).mean().fillna(method="bfill")
C:\Users\redaj\AppData\Local\Temp\ipykernel_19792\3959326269.py:92: FutureWarning: Series.fillna with 'method' is deprecated and will raise 

  ✓ Saved: data/processed/jeddah_processed.csv

  Processing: Abha
  ✓ After daylight filter: 32,880 rows
  ✓ Dropped 0 rows with nulls
  ✓ Final rows: 32,880
  ✓ Train rows: 21,915
  ✓ Val rows:   5,475
  ✓ Test rows:  5,490
  ✓ Seasons: {'Spring': 8280, 'Summer': 8280, 'Autumn': 8190, 'Winter': 8130}


...

  ✓ Saved: data/processed/abha_processed.csv

  Processing: Dammam
  ✓ After daylight filter: 32,880 rows
  ✓ Dropped 0 rows with nulls
  ✓ Final rows: 32,880
  ✓ Train rows: 21,915
  ✓ Val rows:   5,475
  ✓ Test rows:  5,490
  ✓ Seasons: {'Spring': 8280, 'Summer': 8280, 'Autumn': 8190, 'Winter': 8130}


C:\Users\redaj\AppData\Local\Temp\ipykernel_19792\3959326269.py:89: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["cloud_trend_3h"]    = df["cloud_cover"].rolling(3).mean().fillna(method="bfill")
C:\Users\redaj\AppData\Local\Temp\ipykernel_19792\3959326269.py:90: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["wind_trend_3h"]     = df["wind_speed"].rolling(3).mean().fillna(method="bfill")
C:\Users\redaj\AppData\Local\Temp\ipykernel_19792\3959326269.py:91: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["humidity_trend_3h"] = df["humidity"].rolling(3).mean().fillna(method="bfill")
C:\Users\redaj\AppData\Local\Temp\ipykernel_19792\3959326269.py:92: FutureWarning: Series.fillna with 'method' is deprecated and will raise 

  ✓ Saved: data/processed/dammam_processed.csv

  Processing: Qassim
  ✓ After daylight filter: 32,880 rows
  ✓ Dropped 0 rows with nulls
  ✓ Final rows: 32,880
  ✓ Train rows: 21,915
  ✓ Val rows:   5,475
  ✓ Test rows:  5,490
  ✓ Seasons: {'Spring': 8280, 'Summer': 8280, 'Autumn': 8190, 'Winter': 8130}


C:\Users\redaj\AppData\Local\Temp\ipykernel_19792\3959326269.py:89: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["cloud_trend_3h"]    = df["cloud_cover"].rolling(3).mean().fillna(method="bfill")
C:\Users\redaj\AppData\Local\Temp\ipykernel_19792\3959326269.py:90: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["wind_trend_3h"]     = df["wind_speed"].rolling(3).mean().fillna(method="bfill")
C:\Users\redaj\AppData\Local\Temp\ipykernel_19792\3959326269.py:91: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["humidity_trend_3h"] = df["humidity"].rolling(3).mean().fillna(method="bfill")
C:\Users\redaj\AppData\Local\Temp\ipykernel_19792\3959326269.py:92: FutureWarning: Series.fillna with 'method' is deprecated and will raise 

  ✓ Saved: data/processed/qassim_processed.csv

  PREPROCESSING COMPLETE

  Features created:
  PAPER FEATURES:
    hour, month, day_of_year, cloud_cover,
    temperature, humidity, hour_x_cloud,
    temp_x_humidity, clear_sky_ratio

  OUR NEW FEATURES:
    dust_proxy, wind_sin, wind_cos,
    wind_speed_x_cloud, pressure_diff,
    cloud_trend_3h, wind_trend_3h,
    humidity_trend_3h, GHI_trend_3h,
    cloud_change_1h, wind_change_1h,
    GHI_change_1h

  Data split:
    Train → 2019-2022 (4 years)
    Val   → 2023      (1 year)
    Test  → 2024      (1 year)
